# Initial Steps: load packages, files and functions

In [1]:
import json, re, time, itertools
from copy import deepcopy
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

In [3]:

xlsx_path = Path("Altered Meeting history.xlsx")  # <-- change if needed

xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])  # Participants (first sheet)
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df.iloc[ 1:2] = "Targeting Lipid Biology in Cancer"
participants_df.tail(5)

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
2375,"Jeffrey Ward, MD, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Scholar,Jeffrey,Ward,"MD, PhD",Washington University,NaN
2376,"David Barbie, MD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,David,Barbie,MD,Ambrosino Biotech Consulting LLC,Associate Professor of Medicine
2377,"David Shackelford, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,David,Shackelford,PhD,University of California Los Angeles,Professor of Medicine
2378,"Daniel Frigo, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,Daniel,Frigo,PhD,MD Anderson Cancer Center,Associate Professor
2379,"Jessie Yanxiang Guo, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,Jessie Yanxiang,Guo,PhD,Rutgers Cancer Institute,Associate Professor


In [4]:
_TITLE_RE = re.compile(r"^(dr\.?|prof\.?|mr\.?|ms\.?|mrs\.?)\s+", re.I)
# Strips trailing comma-separated credentials (extend list if you have others)
_CRED_RE = re.compile(r"(?:,?\s*(?:MD|M\.D\.|PhD|Ph\.D\.|DO|D\.O\.|MPH|MS|MSc|MBA|JD|DDS|DVM|RN))+$", re.I)

def normalize_meeting_title(x: str) -> str:
    if pd.isna(x): return None
    s = str(x).strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {"'", '"'}:
        s = s[1:-1].strip()
    return re.sub(r"\s+", " ", s)

def strip_titles_and_credentials(name: str) -> str:
    if pd.isna(name): return None
    s = str(name).strip()
    s = _TITLE_RE.sub("", s)      # leading "Dr.", "Prof.", etc.
    s = _CRED_RE.sub("", s)       # trailing ", MD, PhD" etc.
    return re.sub(r"\s+", " ", s).strip()

def split_chair_names(chairs_cell) -> list[str]:
    if pd.isna(chairs_cell): return []
    s = re.sub(r"\s+", " ", str(chairs_cell).strip())
    parts = [p.strip() for p in s.split(";") if p.strip()]
    chairs = []
    for p in parts:
        for item in re.split(r"\s+(?:and|&)\s+", p):
            item = item.strip()
            if not item: 
                continue
            item = re.sub(r"\s+of\s+.+$", "", item).strip()   # drop trailing institution
            item = strip_titles_and_credentials(item)         # drop titles/credentials
            if item:
                chairs.append(item)
    out, seen = [], set()
    for c in chairs:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

In [5]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

# Normalize meeting names
meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Strip titles/credentials from participant names (THIS is the key change)
participants_df["Participant_clean"] = participants_df["Participant"].map(strip_titles_and_credentials)

# Map normalized meeting topic -> year (first non-null year per meeting)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year to participants
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

# Build dict: "Meeting Topic (Year)" -> sorted unique participant_clean names
def make_meeting_year_key(meeting_norm, year):
    if meeting_norm is None:
        return None
    if pd.isna(year):
        return f"{meeting_norm} (Year Unknown)"
    y = int(year) if float(year).is_integer() else year
    return f"{meeting_norm} ({y})"

tmp = participants_df.dropna(subset=["Meeting_norm", "Participant_clean"]).copy()
tmp["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(tmp["Meeting_norm"], tmp["Year"])]

meeting_attendees_dict = (
    tmp.groupby("MeetingYearKey")["Participant_clean"]
       .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
       .to_dict()
)

# optional: ensure chairs are included too (also title/credential stripped)
meetings_df["Chairs_clean"] = meetings_df["Meeting Chairs"].apply(split_chair_names)
for _, row in meetings_df.iterrows():
    topic = row.get("MeetingTopic_norm")
    year = row.get("Year")
    if not topic:
        continue
    key = make_meeting_year_key(topic, year)
    if key not in meeting_attendees_dict:
        meeting_attendees_dict[key] = []
    for chair in (row.get("Chairs_clean") or []):
        meeting_attendees_dict[key].append(chair)
    meeting_attendees_dict[key] = sorted(set(meeting_attendees_dict[key]))

# Preview
list(meeting_attendees_dict.items())[:5]

[('2005 Scholar Retreat (2005)',
  ['Alison Bertuch',
   'Anthony Letai',
   'Charles L. Sawyers',
   'Charles Sherr',
   'Christopher Bakkenist',
   'Craig Thompson',
   'David E. Fisher',
   'David Tuveson',
   'Edward Attiyeh',
   'Elsa Flores',
   'James Amatruda',
   'Jan Karlseder',
   'Kimryn Rathmell',
   'Masashi Narita',
   'Nabeel Bardeesy',
   'Norman Sharpless',
   'Scott Lowe']),
 ('2006 Scholar Retreat (2006)',
  ['Alison Bertuch',
   'Anthony Letai',
   'Benjamin B. Willia',
   'Christopher Bakkenist',
   'Ed Harlow',
   'Edward Attiyeh',
   'Elsa Flores',
   'Gerard Evan',
   'Ingo K. Mellinghoff',
   'James Amatruda',
   'Jan Karlseder',
   'Jean Wang',
   'John Kemshead',
   'Kimberly Kelly',
   'Kimryn Rathmell',
   'Masashi Narita',
   'Michael Safran',
   'Nabeel Bardeesy',
   'Norman Sharpless']),
 ('2007 Scholar Retreat (2007)',
  ['Anthony Letai',
   'Benjamin B. Willia',
   'Benjamin L. Ebert',
   'Carla F. Bender Kim',
   'Catriona Jamieson',
   'Edward Attiy

In [9]:

REPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"
_SUFFIXES = {"jr","sr","ii","iii","iv","md","phd","mph","ms","m.d.","ph.d.","dr"}

def meeting_year_from_key(meeting_key: str):
    m = re.search(r"\((\d{4})\)\s*$", str(meeting_key))
    return int(m.group(1)) if m else None

def split_first_last(name: str):
    s = re.sub(r"\([^)]*\)", "", str(name or "")).strip()
    if not s: return "", ""
    if "," in s:
        last, first = [x.strip() for x in s.split(",", 1)]
        first = first.split()[0] if first else ""
        return first.lower(), last.lower()
    toks = [t for t in s.replace(".", " ").split() if t]
    toks = [t for t in toks if t.lower() not in _SUFFIXES]
    if len(toks) == 1: return "", toks[0].lower()
    return toks[0].lower(), toks[-1].lower()

def reporter_search_all_pages(payload: dict, sleep=0.25, limit=500):
    params = deepcopy(payload); params.update({"offset": 0, "limit": limit})
    pages, total = [], None
    while True:
        r = requests.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json(); pages.append(page)
        total = total or page.get("meta", {}).get("total", 0)
        off = page.get("meta", {}).get("offset", params["offset"])
        cnt = page.get("meta", {}).get("count", len(page.get("results", [])))
        if cnt == 0 or off + cnt >= total: break
        params["offset"] = off + cnt
        time.sleep(sleep)
    return {"total": int(total or 0), "pages": pages}

def extract_pi_names(pi_list):
    # pi_list comes back as PrincipalInvestigators (list of dicts)
    if not isinstance(pi_list, list): return ""
    out = []
    for pi in pi_list:
        if isinstance(pi, dict):
            fn = (pi.get("FirstName") or pi.get("first_name") or "").strip()
            ln = (pi.get("LastName")  or pi.get("last_name")  or "").strip()
            if ln and fn: out.append(f"{ln}, {fn}")
            elif ln: out.append(ln)
            elif fn: out.append(fn)
            else:
                nm = pi.get("Name") or pi.get("PiName") or pi.get("pi_name") or ""
                if nm: out.append(str(nm))
        else:
            out.append(str(pi))
    seen, dedup = set(), []
    for x in out:
        x = re.sub(r"\s+", " ", x).strip()
        if x and x not in seen:
            seen.add(x); dedup.append(x)
    return "; ".join(dedup)

def pick_award_amount(proj: dict) -> int:
    # Prefer AwardAmount; fall back to FyTotalCost if present; else 0
    c = proj.get("AwardAmount")
    if c is None: c = proj.get("FyTotalCost")
    return int(c or 0)

def grants_by_attendee_year_bins(meeting_attendees_dict: dict, first_n_meetings=10, sleep=0.25, verbose=True):
    NIH_param_template = {"criteria": {}}
    rows = []
    meeting_keys = list(meeting_attendees_dict.keys())[:first_n_meetings]

    print(f"Running NIH RePORTER attendee grant search for first {len(meeting_keys)} meetings...")
    for mi, meeting_key in enumerate(meeting_keys, start=1):
        meeting_year = meeting_year_from_key(meeting_key)
        if meeting_year is None:
            if verbose: print(f"[{mi}/{len(meeting_keys)}] {meeting_key}: no year found, skipping")
            continue

        attendees = [n for n in meeting_attendees_dict.get(meeting_key, []) if isinstance(n, str) and n.strip()]
        if verbose:
            print(f"\n[{mi}/{len(meeting_keys)}] Meeting: {meeting_key} | Year={meeting_year} | Attendees={len(attendees)}")

        for y in range(meeting_year - 5, meeting_year + 10 + 1):
            if verbose: print(f"  FY={y} ...")
            for attendee in attendees:
                first, last = split_first_last(attendee)
                if not last: 
                    continue

                payload = deepcopy(NIH_param_template)
                c = payload["criteria"]
                c["pi_names"] = [{"any_name": last, "first_name": first or ""}]
                c["fiscal_years"] = [int(y)]

                # ✅ IMPORTANT: use APIIncludeFieldName (PascalCase)
                payload["include_fields"] = [
                    "ApplId","FiscalYear","AwardAmount","FyTotalCost",
                    "ProjectNum","ProjectTitle","PrincipalInvestigators"
                ]

                res = reporter_search_all_pages(payload, sleep=sleep)
                if res["total"] <= 0:
                    continue

                for page in res["pages"]:
                    for proj in page.get("results", []):
                        appl = proj.get("ApplId")
                        if appl is None:
                            continue
                        rows.append({
                            "MeetingYearKey": meeting_key,
                            "MeetingYear": meeting_year,
                            "BinFiscalYear": int(y),
                            "AttendeeQueried": attendee,
                            "ApplId": appl,
                            "AwardAmount": pick_award_amount(proj),
                            "ProjectNum": proj.get("ProjectNum"),
                            "ProjectTitle": proj.get("ProjectTitle"),
                            "PI_Names_On_Project": extract_pi_names(proj.get("PrincipalInvestigators")),
                        })

                time.sleep(sleep)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["MeetingYearKey","BinFiscalYear","AttendeeQueried","ApplId"])
    return df

In [8]:
attendee_grants_df = grants_by_attendee_year_bins(
    meeting_attendees_dict=meeting_attendees_dict,
    first_n_meetings=10,
    sleep=0.25,
    verbose=True
)

attendee_grants_df.head(25)

Running NIH RePORTER attendee grant search for first 10 meetings...

[1/10] Meeting: 2005 Scholar Retreat (2005) | Year=2005 | Attendees=17
  FY=2000 ...
{'total': 1, 'pages': [{'meta': {'search_id': 'vDuV6QMHD0eu3FRhE2LIuQ', 'total': 1, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/vDuV6QMHD0eu3FRhE2LIuQ/projects'}}, 'results': [{}]}]}
{'total': 4, 'pages': [{'meta': {'search_id': 'jeWWK4PCrE-HogxgZ60zrA', 'total': 4, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/jeWWK4PCrE-HogxgZ60zrA/projects'}}, 'results': [{}, {}, {}, {}]}]}
{'total': 37, 'pages': [{'meta': {'search_id': 's7GpdMz2x02Q8Sf0AWTajA', 'total': 37, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/s7GpdMz2x02Q8Sf0A

KeyboardInterrupt: 

In [ ]:
# How many grants per meeting?
attendee_grants_df.groupby("MeetingYearKey")["ApplId"].nunique()

# How many grants per year bin?
attendee_grants_df.groupby("BinFiscalYear")["ApplId"].nunique()

# Example: all grants for a single attendee
attendee_grants_df[attendee_grants_df["AttendeeQueried"].str.contains("Smith", case=False)]